In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import torch
import numpy as np
import pandas as pd
import os
from transformers import AutoModel, AutoImageProcessor
from tqdm import tqdm
from PIL import Image
from dotenv import load_dotenv

from transformers.utils import logging
logging.disable_progress_bar()

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [ ]:
model_id = "openai/clip-vit-large-patch14"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

processor = AutoImageProcessor.from_pretrained(model_id, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    model_id,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    output_hidden_states=True,
    token=HF_TOKEN
).to(device).eval()
print("model loaded")

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded


In [12]:
model

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [7]:
DEBIAS_LAYER = 15
IMAGES_DIR = "data/images"
METADATA_PATH = "data/metadata.csv"
CAVS_DIR = "models/cavs"
ACT_DM_DIR = "data/activations/dm"
ACT_LR_DIR = "data/activations/lr"

os.makedirs(ACT_DM_DIR, exist_ok=True)
os.makedirs(ACT_LR_DIR, exist_ok=True)

cav_dm = np.load(os.path.join(CAVS_DIR, f"cav_dm_layer_{DEBIAS_LAYER}.npy"))
cav_lr = np.load(os.path.join(CAVS_DIR, f"cav_lr_layer_{DEBIAS_LAYER}.npy"))

df = pd.read_csv(METADATA_PATH)
test_df = df[df['split'] == 'test']

In [ ]:
# Helper to orthogonalize tensor
def project_out(x, v):
    v_tensor = torch.from_numpy(v).to(x.device, dtype=x.dtype)
    dot = (x @ v_tensor)
    res = x - dot.unsqueeze(-1) * v_tensor
    return res

In [9]:
# Debiased inference using hooks, capturing subsequent layers
def extract_debiased_activations(df_subset, cav, target_layer_idx):
    all_layer_acts = []
    labels = []
    layer_outputs = {}
    handles = []
    
    def debias_hook(module, input, output):
        is_tuple = isinstance(output, tuple)
        hidden_states = output[0] if is_tuple else output
        
        # apply orthogonalization on ALL tokens
        hidden_states = project_out(hidden_states, cav)
        
        layer_outputs[target_layer_idx] = hidden_states[:, 0, :].detach().cpu()
        if is_tuple:
            return (hidden_states,) + output[1:]
        return hidden_states

    def make_capture_hook(idx):
        def capture_hook(module, input, output):
            is_tuple = isinstance(output, tuple)
            hidden_states = output[0] if is_tuple else output
            layer_outputs[idx] = hidden_states[:, 0, :].detach().cpu()
        return capture_hook

    encoder_layers = model.vision_model.encoder.layers
    handles.append(encoder_layers[target_layer_idx].register_forward_hook(debias_hook))
    
    for i in range(target_layer_idx + 1, len(encoder_layers)):
        handles.append(encoder_layers[i].register_forward_hook(make_capture_hook(i)))
        
    try:
        with torch.no_grad():
            for _, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
                img_path = os.path.join(IMAGES_DIR, row['filename'])
                img = Image.open(img_path).convert("RGB")
                inputs = processor(images=img, return_tensors="pt").to(device, dtype=dtype)
                
                model.get_image_features(**inputs)
                
                img_acts = []
                for i in range(target_layer_idx, len(encoder_layers)):
                    img_acts.append(layer_outputs[i].float().numpy().squeeze())
                all_layer_acts.append(img_acts)
                labels.append(row['bald'])
    finally:
        for h in handles:
            h.remove()
            
    return np.array(all_layer_acts), np.array(labels)

In [10]:
# Export activations to CSV files layer by layer
def save_activations(acts, labels, cav_type, target_layer_idx):
    n_layers = acts.shape[1]
    for i in range(n_layers):
        layer_num = target_layer_idx + i
        layer_data = acts[:, i, :]
        df_acts = pd.DataFrame(layer_data)
        df_acts['label'] = labels
        if cav_type == 'dm':
            save_path = os.path.join(ACT_DM_DIR, f"layer_{layer_num}.csv")
        else:
            save_path = os.path.join(ACT_LR_DIR, f"layer_{layer_num}.csv")
        df_acts.to_csv(save_path, index=False)

In [11]:
print("Running analysis with Diff Means CAV...")
results_dm, labels_test = extract_debiased_activations(test_df, cav_dm, DEBIAS_LAYER)
save_activations(results_dm, labels_test, 'dm', DEBIAS_LAYER)

print("Running analysis with Logistic Regression CAV...")
results_lr, _ = extract_debiased_activations(test_df, cav_lr, DEBIAS_LAYER)
save_activations(results_lr, labels_test, 'lr', DEBIAS_LAYER)

print("Debiased inference complete and activations saved.")

Running analysis with Diff Means CAV...


100%|██████████| 2000/2000 [06:44<00:00,  4.94it/s]


Running analysis with Logistic Regression CAV...


100%|██████████| 2000/2000 [06:44<00:00,  4.94it/s]


Debiased inference complete and activations saved.
